# Advanced Problems with Solutions: Inheriting from `type`

This notebook develops advanced skill with **custom metaclasses created by subclassing `type`**.

You will work with:

- `type.__new__` and cooperative `super()`
- namespace transformation during class creation
- method injection
- validation and class contracts
- automatic class registries
- metaclass `__call__`
- `__prepare__`
- descriptors created by a metaclass
- inherited configuration
- abstract base classes and metaclass composition
- metaclass conflicts
- class decorators vs `__init_subclass__` vs metaclasses
- testing metaclass behavior
- common mistakes and production best practices

Every problem includes a complete solution and executable checks.

> **Best-practice theme:** Metaclasses are powerful, but they should be used only when class creation itself must be customized. Prefer simpler mechanisms such as class decorators or `__init_subclass__` when they solve the problem cleanly.


## 0. Mental model refresher

A normal class definition:

```python
class Circle:
    ...
```

ultimately creates a **class object**. By default, that class object is an instance of `type`.

A custom metaclass is usually a subclass of `type`:

```python
class MyMeta(type):
    ...
```

Then:

```python
class Example(metaclass=MyMeta):
    ...
```

causes Python to use `MyMeta` to construct the `Example` class object.

The most important creation hooks are:

1. `metaclass.__prepare__(name, bases, **kwargs)` → returns the namespace mapping used while executing the class body.
2. `metaclass.__new__(mcls, name, bases, namespace, **kwargs)` → creates the class object.
3. `metaclass.__init__(cls, name, bases, namespace, **kwargs)` → initializes the already-created class object.
4. `metaclass.__call__(cls, *args, **kwargs)` → controls what happens when the resulting class is called to create instances.

For most transformations, `__new__` is the main hook.


In [1]:
import math
from abc import ABC, ABCMeta, abstractmethod
from collections import OrderedDict
from types import MappingProxyType


## 1. Warm-up — injecting behavior safely

### Problem

Create a metaclass `GeometryMeta` that injects a method named `circumference` into any class that:

- has no method named `circumference` already, and
- stores a radius in `self.r`.

The injected method should return `2 * pi * r`.

Do **not** overwrite a user-defined `circumference`.


In [2]:
class GeometryMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        namespace = dict(namespace)  # defensive copy

        if "circumference" not in namespace:
            def circumference(self):
                return 2 * math.pi * self.r

            namespace["circumference"] = circumference

        return super().__new__(mcls, name, bases, namespace, **kwargs)


class Circle(metaclass=GeometryMeta):
    def __init__(self, r):
        self.r = r

    def area(self):
        return math.pi * self.r ** 2


c = Circle(3)
assert math.isclose(c.circumference(), 6 * math.pi)
assert math.isclose(c.area(), 9 * math.pi)
type(Circle), c.circumference()


(__main__.GeometryMeta, 18.84955592153876)

### Why this version is better than a quick demo

A robust metaclass generally:

- accepts `**kwargs` so class keywords can be forwarded,
- uses `super()` cooperatively,
- avoids silently overwriting methods supplied by the class author,
- keeps injected functions readable and testable,
- minimizes hidden behavior.


## 2. Problem — enforce a class-level contract

### Problem

Build `ValidatedModelMeta`.

Every concrete class using this metaclass must define:

```python
fields = ("field_a", "field_b", ...)
```

Requirements:

1. `fields` must exist.
2. It must be a tuple.
3. Every entry must be a valid identifier.
4. Duplicate field names are forbidden.
5. Private names beginning with `_` are forbidden.
6. The base helper class `ModelBase` itself is exempt.

Raise clear exceptions during **class creation**, not during instance creation.


In [3]:
class ValidatedModelMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        cls = super().__new__(mcls, name, bases, namespace, **kwargs)

        # Exempt the framework base class.
        if name == "ModelBase" and not bases:
            return cls

        if "fields" not in namespace:
            raise TypeError(f"{name} must define a 'fields' tuple")

        fields = namespace["fields"]

        if not isinstance(fields, tuple):
            raise TypeError(f"{name}.fields must be a tuple")

        if len(fields) != len(set(fields)):
            raise ValueError(f"{name}.fields contains duplicates")

        for field in fields:
            if not isinstance(field, str):
                raise TypeError("field names must be strings")
            if not field.isidentifier():
                raise ValueError(f"{field!r} is not a valid Python identifier")
            if field.startswith("_"):
                raise ValueError(f"private field names are forbidden: {field!r}")

        return cls


class ModelBase(metaclass=ValidatedModelMeta):
    pass


class User(ModelBase):
    fields = ("name", "email", "age")


User.fields


('name', 'email', 'age')

In [4]:
def expect_class_creation_error(source, expected_exception):
    try:
        exec(source, globals(), {})
    except expected_exception as exc:
        return type(exc).__name__, str(exc)
    else:
        raise AssertionError(f"Expected {expected_exception.__name__}")


checks = [
    expect_class_creation_error(
        "class BadMissing(ModelBase):\n    pass",
        TypeError,
    ),
    expect_class_creation_error(
        "class BadList(ModelBase):\n    fields = ['x']",
        TypeError,
    ),
    expect_class_creation_error(
        "class BadDuplicate(ModelBase):\n    fields = ('x', 'x')",
        ValueError,
    ),
    expect_class_creation_error(
        "class BadIdentifier(ModelBase):\n    fields = ('not valid',)",
        ValueError,
    ),
]

checks


[('TypeError', "BadMissing must define a 'fields' tuple"),
 ('TypeError', 'BadList.fields must be a tuple'),
 ('ValueError', 'BadDuplicate.fields contains duplicates'),
 ('ValueError', "'not valid' is not a valid Python identifier")]

### Discussion

Class-creation validation is valuable when invalid classes should never exist.

This is preferable to discovering the error much later when an instance is constructed or a method is called.


## 3. Problem — generate properties automatically

### Problem

Extend the previous idea.

Create `FieldsMeta` such that:

```python
class Point(metaclass=FieldsMeta):
    fields = ("x", "y")
```

automatically receives:

- an `__init__(self, x, y)` method,
- read/write properties `x` and `y`,
- a useful `__repr__`.

Store values internally as `_x`, `_y`, etc.

Reject callers that pass the wrong number of positional arguments.


In [5]:
class FieldsMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        fields = namespace.get("fields", ())

        if not isinstance(fields, tuple):
            raise TypeError("fields must be a tuple")

        for field in fields:
            if not isinstance(field, str) or not field.isidentifier():
                raise ValueError(f"Invalid field name: {field!r}")

        namespace = dict(namespace)

        if "__init__" not in namespace:
            def __init__(self, *values):
                if len(values) != len(fields):
                    raise TypeError(
                        f"{type(self).__name__} expected {len(fields)} values, "
                        f"got {len(values)}"
                    )
                for field, value in zip(fields, values):
                    setattr(self, f"_{field}", value)

            namespace["__init__"] = __init__

        for field in fields:
            if field not in namespace:
                def getter(self, _field=field):
                    return getattr(self, f"_{_field}")

                def setter(self, value, _field=field):
                    setattr(self, f"_{_field}", value)

                namespace[field] = property(getter, setter)

        if "__repr__" not in namespace:
            def __repr__(self):
                values = ", ".join(
                    f"{field}={getattr(self, field)!r}" for field in fields
                )
                return f"{type(self).__name__}({values})"

            namespace["__repr__"] = __repr__

        return super().__new__(mcls, name, bases, namespace, **kwargs)


class Point(metaclass=FieldsMeta):
    fields = ("x", "y")


p = Point(10, 20)
assert p.x == 10 and p.y == 20
p.x = 99
assert p.x == 99
repr(p)


'Point(x=99, y=20)'

### Important closure detail

The getter and setter use default parameters such as `_field=field`.

Without that technique, all generated functions would close over the same loop variable and often end up referring to the **last** field.


## 4. Problem — automatic plugin registry

### Problem

Create a `PluginMeta` metaclass.

Every non-abstract plugin class should be registered by a unique string key:

```python
plugin_name = "json"
```

Requirements:

- the registry lives on the metaclass,
- duplicate names raise an error,
- classes can be retrieved by `PluginMeta.get("json")`,
- the framework base class is not registered,
- a plugin can opt out with `abstract = True`.


In [6]:
class PluginMeta(type):
    _registry = {}

    def __new__(mcls, name, bases, namespace, **kwargs):
        cls = super().__new__(mcls, name, bases, namespace, **kwargs)

        is_framework_base = not bases
        is_abstract = namespace.get("abstract", False)

        if not is_framework_base and not is_abstract:
            plugin_name = namespace.get("plugin_name")

            if not isinstance(plugin_name, str) or not plugin_name:
                raise TypeError(
                    f"{name} must define a non-empty string 'plugin_name'"
                )

            if plugin_name in mcls._registry:
                existing = mcls._registry[plugin_name]
                raise ValueError(
                    f"Duplicate plugin name {plugin_name!r}: "
                    f"{existing.__name__} already registered"
                )

            mcls._registry[plugin_name] = cls

        return cls

    @classmethod
    def get(mcls, plugin_name):
        return mcls._registry[plugin_name]

    @classmethod
    def registry(mcls):
        return MappingProxyType(mcls._registry)


class Plugin(metaclass=PluginMeta):
    abstract = True


class JSONPlugin(Plugin):
    abstract = False
    plugin_name = "json"

    def parse(self, text):
        import json
        return json.loads(text)


class TextPlugin(Plugin):
    abstract = False
    plugin_name = "text"

    def parse(self, text):
        return text


assert PluginMeta.get("json") is JSONPlugin
assert PluginMeta.get("text") is TextPlugin
dict(PluginMeta.registry())


{'json': __main__.JSONPlugin, 'text': __main__.TextPlugin}

### Best-practice note

Returning a `MappingProxyType` gives callers a read-only view of the registry, reducing accidental mutation.


## 5. Problem — control instance creation with metaclass `__call__`

### Problem

Implement `SingletonMeta`.

For every class that uses this metaclass:

- calling the class repeatedly returns the same instance,
- each class gets its **own** singleton instance,
- instance creation still delegates to the normal machinery.

Then explain why this belongs in `__call__`, not `__new__` of the metaclass.


In [7]:
class SingletonMeta(type):
    _instances = {}

    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]


class Config(metaclass=SingletonMeta):
    def __init__(self, value):
        self.value = value


class Cache(metaclass=SingletonMeta):
    def __init__(self):
        self.data = {}


a = Config(10)
b = Config(999)
cache1 = Cache()
cache2 = Cache()

assert a is b
assert a.value == 10
assert cache1 is cache2
assert a is not cache1

(a is b, cache1 is cache2, a.value)


(True, True, 10)

### Why `__call__`?

`SingletonMeta.__new__` controls creation of the **class object** (`Config`).

`SingletonMeta.__call__` controls what happens when that class object is called:

```python
Config(...)
```

That is exactly the instance-creation boundary we want to intercept.


## 6. Problem — trace class creation order

### Problem

Write `TracingMeta` that records the order in which:

- `__prepare__`,
- `__new__`,
- `__init__`

are invoked while defining a class.

Then demonstrate that class-body execution occurs after `__prepare__` but before `__new__`.


In [8]:
events = []


class TracingMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        events.append(("prepare", name))
        return {}

    def __new__(mcls, name, bases, namespace, **kwargs):
        events.append(("new", name, tuple(namespace.keys())))
        return super().__new__(mcls, name, bases, namespace, **kwargs)

    def __init__(cls, name, bases, namespace, **kwargs):
        events.append(("init", name))
        super().__init__(name, bases, namespace, **kwargs)


class Demo(metaclass=TracingMeta):
    events.append(("class-body", "Demo"))
    x = 10

    def method(self):
        return self.x


events


[('prepare', 'Demo'),
 ('class-body', 'Demo'),
 ('new',
  'Demo',
  ('__module__',
   '__qualname__',
   '__firstlineno__',
   'x',
   'method',
   '__static_attributes__')),
 ('init', 'Demo')]

In [9]:
labels = [event[0] for event in events]
assert labels == ["prepare", "class-body", "new", "init"]
labels


['prepare', 'class-body', 'new', 'init']

## 7. Problem — use `__prepare__` to detect duplicate definitions

### Problem

Normally, writing the same attribute twice in a class body silently overwrites the first value:

```python
class Example:
    x = 1
    x = 2
```

Create:

1. a custom mapping `NoDuplicatesDict`,
2. a metaclass `NoDuplicatesMeta` whose `__prepare__` returns that mapping.

The class should reject duplicate names **except** for a short allow-list of implementation details such as `__module__` and `__qualname__`.


In [10]:
class NoDuplicatesDict(dict):
    allowed_reassignments = {"__module__", "__qualname__", "__annotations__"}

    def __setitem__(self, key, value):
        if key in self and key not in self.allowed_reassignments:
            raise TypeError(f"Duplicate class attribute: {key!r}")
        super().__setitem__(key, value)


class NoDuplicatesMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        return NoDuplicatesDict()


class Good(metaclass=NoDuplicatesMeta):
    x = 10

    def method(self):
        return self.x


assert Good().method() == 10

try:
    exec(
        '''
class Bad(metaclass=NoDuplicatesMeta):
    x = 1
    x = 2
''',
        globals(),
        {},
    )
except TypeError as exc:
    duplicate_error = str(exc)

duplicate_error


"Duplicate class attribute: 'x'"

### Why `__prepare__` is the right hook

Duplicate information is lost by the time `__new__` receives a normal dictionary because the later assignment has already replaced the earlier one.

A custom class namespace lets you observe assignments **as the class body executes**.


## 8. Problem — automatically wrap public methods

### Problem

Create `LoggingMeta`.

For every user-defined public instance method, wrap it so calls are appended to a shared `call_log` list.

Do not wrap:

- dunder methods,
- private methods whose name starts with `_`,
- `staticmethod`,
- `classmethod`,
- non-callable attributes.

Preserve the wrapped function's metadata using `functools.wraps`.


In [11]:
import functools

call_log = []


class LoggingMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        namespace = dict(namespace)

        def make_wrapper(method_name, func):
            @functools.wraps(func)
            def wrapper(self, *args, **kwargs):
                call_log.append(
                    (type(self).__name__, method_name, args, dict(kwargs))
                )
                return func(self, *args, **kwargs)

            return wrapper

        for attr_name, attr_value in list(namespace.items()):
            if attr_name.startswith("_"):
                continue

            if isinstance(attr_value, (staticmethod, classmethod)):
                continue

            if callable(attr_value):
                namespace[attr_name] = make_wrapper(attr_name, attr_value)

        return super().__new__(mcls, name, bases, namespace, **kwargs)


class Calculator(metaclass=LoggingMeta):
    version = 1

    def add(self, a, b):
        return a + b

    def multiply(self, a, b):
        return a * b

    def _helper(self):
        return "private"

    @staticmethod
    def identity(x):
        return x


calc = Calculator()
assert calc.add(2, 3) == 5
assert calc.multiply(4, 5) == 20
assert Calculator.identity(7) == 7

call_log


[('Calculator', 'add', (2, 3), {}), ('Calculator', 'multiply', (4, 5), {})]

### Production caveat

Method wrapping at class creation can make debugging and introspection more complex. Prefer explicit decorators when only a few methods need instrumentation.


## 9. Problem — descriptor generation with validation

### Problem

Create a descriptor `TypedField` and a metaclass `TypedMeta`.

A class may declare:

```python
schema = {
    "name": str,
    "age": int,
}
```

The metaclass should replace each schema entry with a descriptor that:

- accepts values of the declared type,
- raises `TypeError` otherwise,
- stores data under a private attribute,
- knows its attribute name via `__set_name__`.

The metaclass should also generate an `__init__` accepting keyword arguments.


In [12]:
class TypedField:
    def __init__(self, expected_type):
        self.expected_type = expected_type
        self.name = None

    def __set_name__(self, owner, name):
        self.name = name
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return getattr(instance, self.storage_name)

    def __set__(self, instance, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(
                f"{self.name} must be {self.expected_type.__name__}, "
                f"got {type(value).__name__}"
            )
        setattr(instance, self.storage_name, value)


class TypedMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        namespace = dict(namespace)
        schema = namespace.get("schema", {})

        if not isinstance(schema, dict):
            raise TypeError("schema must be a dict")

        for field_name, expected_type in schema.items():
            if not isinstance(expected_type, type):
                raise TypeError(
                    f"schema value for {field_name!r} must be a type"
                )
            namespace[field_name] = TypedField(expected_type)

        if schema and "__init__" not in namespace:
            def __init__(self, **values):
                missing = set(schema) - set(values)
                extra = set(values) - set(schema)

                if missing:
                    raise TypeError(f"Missing fields: {sorted(missing)}")
                if extra:
                    raise TypeError(f"Unexpected fields: {sorted(extra)}")

                for field_name, value in values.items():
                    setattr(self, field_name, value)

            namespace["__init__"] = __init__

        return super().__new__(mcls, name, bases, namespace, **kwargs)


class Person(metaclass=TypedMeta):
    schema = {
        "name": str,
        "age": int,
    }


person = Person(name="Ada", age=36)
assert person.name == "Ada"
assert person.age == 36

try:
    person.age = "thirty-six"
except TypeError as exc:
    typed_error = str(exc)

typed_error


'age must be int, got str'

## 10. Problem — inheritance-aware schemas

### Problem

Improve the previous metaclass.

A subclass should automatically inherit schema fields from base classes and may add new fields.

Example:

```python
class Entity(...):
    schema = {"id": int}

class Customer(Entity):
    schema = {"name": str}
```

`Customer` should effectively have:

```python
{"id": int, "name": str}
```

Do not mutate the base class's schema.


In [13]:
class InheritedTypedMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        namespace = dict(namespace)

        merged_schema = {}

        for base in bases:
            base_schema = getattr(base, "schema", {})
            merged_schema.update(base_schema)

        local_schema = namespace.get("schema", {})
        if not isinstance(local_schema, dict):
            raise TypeError("schema must be a dict")

        merged_schema.update(local_schema)
        namespace["schema"] = merged_schema

        # Add descriptors only for locally new/replaced fields.
        for field_name, expected_type in local_schema.items():
            if not isinstance(expected_type, type):
                raise TypeError(f"{field_name!r} schema value must be a type")
            namespace[field_name] = TypedField(expected_type)

        if merged_schema and "__init__" not in namespace:
            def __init__(self, **values):
                expected = set(merged_schema)
                actual = set(values)

                if actual != expected:
                    missing = sorted(expected - actual)
                    extra = sorted(actual - expected)
                    raise TypeError(
                        f"Incorrect fields; missing={missing}, extra={extra}"
                    )

                for field_name, value in values.items():
                    setattr(self, field_name, value)

            namespace["__init__"] = __init__

        return super().__new__(mcls, name, bases, namespace, **kwargs)


class Entity(metaclass=InheritedTypedMeta):
    schema = {"id": int}


class Customer(Entity):
    schema = {"name": str}


customer = Customer(id=1, name="Grace")
assert customer.id == 1
assert customer.name == "Grace"
assert Entity.schema == {"id": int}
assert Customer.schema == {"id": int, "name": str}

Entity.schema, Customer.schema


({'id': int}, {'id': int, 'name': str})

## 11. Problem — metaclass configuration via class keywords

### Problem

Support syntax like:

```python
class APIModel(metaclass=ConfigMeta, table="api_models", frozen=True):
    ...
```

The metaclass should:

- consume `table` and `frozen`,
- store them on the created class as `__table__` and `__frozen__`,
- reject unknown class keywords,
- still cooperate with `type`.

This problem demonstrates why metaclass hooks often need `**kwargs`.


In [14]:
class ConfigMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        allowed = {"table", "frozen"}
        unknown = set(kwargs) - allowed

        if unknown:
            raise TypeError(f"Unknown class options: {sorted(unknown)}")

        table = kwargs.pop("table", name.lower())
        frozen = kwargs.pop("frozen", False)

        cls = super().__new__(mcls, name, bases, namespace, **kwargs)
        cls.__table__ = table
        cls.__frozen__ = bool(frozen)
        return cls


class APIModel(metaclass=ConfigMeta, table="api_models", frozen=True):
    pass


(APIModel.__table__, APIModel.__frozen__)


('api_models', True)

## 12. Problem — combine a custom metaclass with `ABCMeta`

### Problem

Suppose a framework needs both:

- abstract methods from `abc.ABC`,
- registration behavior from a custom metaclass.

A direct combination can create metaclass conflicts.

Create `RegisteredABCMeta` by inheriting from `ABCMeta`, and register only concrete subclasses.


In [15]:
class RegisteredABCMeta(ABCMeta):
    registry = {}

    def __new__(mcls, name, bases, namespace, **kwargs):
        cls = super().__new__(mcls, name, bases, namespace, **kwargs)

        # __abstractmethods__ is established by ABCMeta.
        if not cls.__abstractmethods__ and bases:
            mcls.registry[name] = cls

        return cls


class Serializer(ABC, metaclass=RegisteredABCMeta):
    @abstractmethod
    def dumps(self, obj):
        raise NotImplementedError


class StringSerializer(Serializer):
    def dumps(self, obj):
        return str(obj)


assert "Serializer" not in RegisteredABCMeta.registry
assert RegisteredABCMeta.registry["StringSerializer"] is StringSerializer
dict(RegisteredABCMeta.registry)


{'StringSerializer': __main__.StringSerializer}

### Why this works

`ABC` uses `ABCMeta`.

Because `RegisteredABCMeta` inherits from `ABCMeta`, it is compatible with the abstract-base-class machinery while adding new behavior.


## 13. Problem — reproduce and solve a metaclass conflict

### Problem

Create two unrelated metaclasses and two base classes using them.

Show that multiple inheritance from those base classes causes a metaclass conflict.

Then solve the conflict with a combined metaclass.


In [16]:
class MetaA(type):
    pass


class MetaB(type):
    pass


class BaseA(metaclass=MetaA):
    pass


class BaseB(metaclass=MetaB):
    pass


try:
    class Broken(BaseA, BaseB):
        pass
except TypeError as exc:
    metaclass_conflict_message = str(exc)

metaclass_conflict_message


'metaclass conflict: the metaclass of a derived class must be a (non-strict) subclass of the metaclasses of all its bases'

In [17]:
class CombinedMeta(MetaA, MetaB):
    pass


class Working(BaseA, BaseB, metaclass=CombinedMeta):
    pass


assert isinstance(Working, MetaA)
assert isinstance(Working, MetaB)
type(Working)


__main__.CombinedMeta

### Rule of thumb

For a derived class, Python needs a metaclass that is a subclass of the metaclasses of all base classes.

If no such most-derived metaclass can be selected automatically, you must provide a compatible combined metaclass.


## 14. Problem — prevent subclassing of "final" classes

### Problem

Implement a metaclass `FinalMeta`.

A class can declare:

```python
final = True
```

Any attempt to subclass that class should fail at **class creation time**.

Subclasses that are not final remain subclassable.


In [18]:
class FinalMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        for base in bases:
            if getattr(base, "__is_final__", False):
                raise TypeError(f"Cannot subclass final class {base.__name__}")

        cls = super().__new__(mcls, name, bases, namespace, **kwargs)
        cls.__is_final__ = bool(namespace.get("final", False))
        return cls


class Normal(metaclass=FinalMeta):
    pass


class StillNormal(Normal):
    pass


class Terminal(metaclass=FinalMeta):
    final = True


try:
    class Illegal(Terminal):
        pass
except TypeError as exc:
    final_error = str(exc)

final_error


'Cannot subclass final class Terminal'

## 15. Problem — immutable public class namespace after creation

### Problem

Create a metaclass `SealedMeta` that allows a class body to define attributes normally, but after class creation:

- existing public class attributes may be changed,
- **new** public class attributes may not be added,
- private/dunder attributes may still be added.

This requires overriding metaclass `__setattr__`.


In [19]:
class SealedMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        cls = super().__new__(mcls, name, bases, namespace, **kwargs)
        type.__setattr__(cls, "__sealed__", True)
        return cls

    def __setattr__(cls, name, value):
        is_public = not name.startswith("_")
        is_new = name not in cls.__dict__
        sealed = cls.__dict__.get("__sealed__", False)

        if sealed and is_public and is_new:
            raise AttributeError(
                f"Cannot add new public class attribute {name!r}"
            )

        super().__setattr__(name, value)


class Settings(metaclass=SealedMeta):
    timeout = 30


Settings.timeout = 60  # existing public name: allowed
assert Settings.timeout == 60

try:
    Settings.retries = 5
except AttributeError as exc:
    sealed_error = str(exc)

sealed_error


"Cannot add new public class attribute 'retries'"

## 16. Problem — class factory versus custom metaclass

### Problem

Recreate a simple class in two ways:

1. directly using `type(name, bases, namespace)`,
2. using a custom metaclass subclass.

Compare `type(...)` of the resulting classes.


In [20]:
def greet(self):
    return f"Hello, {self.name}!"


PlainGreeter = type(
    "PlainGreeter",
    (),
    {
        "__init__": lambda self, name: setattr(self, "name", name),
        "greet": greet,
    },
)


class MarkerMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        namespace = dict(namespace)
        namespace["created_by_marker_meta"] = True
        return super().__new__(mcls, name, bases, namespace, **kwargs)


MarkedGreeter = MarkerMeta(
    "MarkedGreeter",
    (),
    {
        "__init__": lambda self, name: setattr(self, "name", name),
        "greet": greet,
    },
)


plain = PlainGreeter("Alice")
marked = MarkedGreeter("Bob")

assert plain.greet() == "Hello, Alice!"
assert marked.greet() == "Hello, Bob!"
assert type(PlainGreeter) is type
assert type(MarkedGreeter) is MarkerMeta

type(PlainGreeter), type(MarkedGreeter), MarkedGreeter.created_by_marker_meta


(type, __main__.MarkerMeta, True)

## 17. Problem — preserve user overrides during injection

### Problem

A common metaclass bug is blindly replacing methods.

Create `SafeReprMeta` that injects a default `__repr__` only when a class body did not define one.

Subclasses should be free to override it.


In [21]:
class SafeReprMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        namespace = dict(namespace)

        if "__repr__" not in namespace:
            def __repr__(self):
                attrs = ", ".join(
                    f"{k}={v!r}"
                    for k, v in sorted(self.__dict__.items())
                )
                return f"{type(self).__name__}({attrs})"

            namespace["__repr__"] = __repr__

        return super().__new__(mcls, name, bases, namespace, **kwargs)


class AutoRepr(metaclass=SafeReprMeta):
    def __init__(self, x):
        self.x = x


class CustomRepr(AutoRepr):
    def __repr__(self):
        return f"<custom x={self.x}>"


a = AutoRepr(10)
b = CustomRepr(20)

assert repr(a) == "AutoRepr(x=10)"
assert repr(b) == "<custom x=20>"

repr(a), repr(b)


('AutoRepr(x=10)', '<custom x=20>')

## 18. Problem — enforce method signatures

### Problem

Create `HandlerMeta`.

Every concrete handler must define:

```python
handle(self, event)
```

with exactly those two positional parameters and no variadic `*args` or `**kwargs`.

Use `inspect.signature`.


In [22]:
import inspect


class HandlerMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        cls = super().__new__(mcls, name, bases, namespace, **kwargs)

        if name == "HandlerBase":
            return cls

        method = namespace.get("handle")
        if method is None:
            raise TypeError(f"{name} must define handle(self, event)")

        signature = inspect.signature(method)
        params = list(signature.parameters.values())

        valid = (
            len(params) == 2
            and params[0].name == "self"
            and params[1].name == "event"
            and all(
                p.kind in (
                    inspect.Parameter.POSITIONAL_ONLY,
                    inspect.Parameter.POSITIONAL_OR_KEYWORD,
                )
                for p in params
            )
        )

        if not valid:
            raise TypeError(
                f"{name}.handle must have signature handle(self, event), "
                f"got {signature}"
            )

        return cls


class HandlerBase(metaclass=HandlerMeta):
    pass


class PrintHandler(HandlerBase):
    def handle(self, event):
        return f"handled: {event}"


assert PrintHandler().handle("created") == "handled: created"

try:
    exec(
        '''
class BadHandler(HandlerBase):
    def handle(self, event, extra=None):
        pass
''',
        globals(),
        {},
    )
except TypeError as exc:
    signature_error = str(exc)

signature_error


'BadHandler.handle must have signature handle(self, event), got (self, event, extra=None)'

## 19. Problem — compare metaclass, `__init_subclass__`, and class decorator

### Problem

The goal is to add every subclass to a registry.

Implement the same behavior in three styles:

1. metaclass,
2. `__init_subclass__`,
3. class decorator.

This is an architectural comparison problem: the code should make it obvious that a metaclass is not always necessary.


In [23]:
# 1) Metaclass approach
class RegistryMeta(type):
    items = []

    def __new__(mcls, name, bases, namespace, **kwargs):
        cls = super().__new__(mcls, name, bases, namespace, **kwargs)
        if bases:
            mcls.items.append(cls)
        return cls


class MetaBase(metaclass=RegistryMeta):
    pass


class MetaChild(MetaBase):
    pass


# 2) __init_subclass__ approach
class InitSubclassBase:
    items = []

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        InitSubclassBase.items.append(cls)


class InitChild(InitSubclassBase):
    pass


# 3) Class decorator approach
decorated_items = []


def register(cls):
    decorated_items.append(cls)
    return cls


@register
class DecoratedChild:
    pass


(
    [cls.__name__ for cls in RegistryMeta.items],
    [cls.__name__ for cls in InitSubclassBase.items],
    [cls.__name__ for cls in decorated_items],
)


(['MetaChild'], ['InitChild'], ['DecoratedChild'])

### Architectural takeaway

Prefer:

- **class decorators** for one-off post-processing of explicitly decorated classes,
- **`__init_subclass__`** for behavior naturally tied to subclasses of one base class,
- **metaclasses** when you truly need to control class construction, custom namespaces, instance creation at the class level, or coordinate framework-wide class semantics.


## 20. Problem — cooperative multiple metaclasses

### Problem

Create two cooperative metaclasses:

- `NameMeta` adds `qualified_label`,
- `VersionMeta` adds `version = 1` when absent.

Combine them into `FrameworkMeta`.

Each metaclass must use `super()` so both transformations run.


In [24]:
class NameMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        namespace = dict(namespace)
        namespace["qualified_label"] = f"{namespace.get('__module__')}.{name}"
        return super().__new__(mcls, name, bases, namespace, **kwargs)


class VersionMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        namespace = dict(namespace)
        namespace.setdefault("version", 1)
        return super().__new__(mcls, name, bases, namespace, **kwargs)


class FrameworkMeta(NameMeta, VersionMeta):
    pass


class Service(metaclass=FrameworkMeta):
    pass


class ServiceV2(metaclass=FrameworkMeta):
    version = 2


assert Service.version == 1
assert ServiceV2.version == 2
assert Service.qualified_label.endswith(".Service")

(Service.qualified_label, Service.version, ServiceV2.version)


('__main__.Service', 1, 2)

### Why cooperative `super()` matters

If `NameMeta.__new__` directly called `type.__new__`, it could skip `VersionMeta.__new__`.

Metaclasses participate in normal Python MRO rules. Framework metaclasses should usually be designed cooperatively.


## 21. Problem — advanced mini-framework

### Problem

Build a small command framework.

Each concrete command class must declare:

```python
command = "name"
```

The metaclass should:

1. validate the command name,
2. reject duplicate command names,
3. require a callable `run(self, **kwargs)` method,
4. register the class,
5. provide `Command.create(name, **kwargs)` to instantiate a command by name.

Use an abstract base class so the framework base itself is not registered.


In [25]:
class CommandMeta(ABCMeta):
    registry = {}

    def __new__(mcls, name, bases, namespace, **kwargs):
        cls = super().__new__(mcls, name, bases, namespace, **kwargs)

        # Only register concrete subclasses.
        if cls.__abstractmethods__:
            return cls

        command_name = namespace.get("command")

        if not isinstance(command_name, str) or not command_name.isidentifier():
            raise TypeError(
                f"{name}.command must be a valid identifier string"
            )

        if command_name in mcls.registry:
            raise ValueError(f"Duplicate command: {command_name!r}")

        run_method = getattr(cls, "run", None)
        if not callable(run_method):
            raise TypeError(f"{name} must provide callable run()")

        mcls.registry[command_name] = cls
        return cls


class Command(ABC, metaclass=CommandMeta):
    @abstractmethod
    def run(self, **kwargs):
        raise NotImplementedError

    @classmethod
    def create(cls, name, **kwargs):
        command_cls = CommandMeta.registry[name]
        return command_cls(**kwargs)


class Echo(Command):
    command = "echo"

    def __init__(self, prefix=""):
        self.prefix = prefix

    def run(self, **kwargs):
        return self.prefix + kwargs["text"]


class Add(Command):
    command = "add"

    def run(self, **kwargs):
        return kwargs["a"] + kwargs["b"]


echo = Command.create("echo", prefix="> ")
adder = Command.create("add")

assert echo.run(text="hello") == "> hello"
assert adder.run(a=2, b=5) == 7

sorted(CommandMeta.registry)


['add', 'echo']

## 22. Problem — test class-creation failures cleanly

### Problem

When metaclass validation intentionally rejects invalid class definitions, tests should assert those failures directly.

Write a reusable helper `assert_class_definition_fails` that:

- executes a class definition from a string,
- checks the exception type,
- optionally checks that a message fragment appears.

Then test it against `FinalMeta`.


In [26]:
def assert_class_definition_fails(
    source,
    exception_type=Exception,
    message_contains=None,
    global_ns=None,
):
    namespace = globals() if global_ns is None else global_ns

    try:
        exec(source, namespace, {})
    except exception_type as exc:
        if message_contains is not None:
            assert message_contains in str(exc), (
                f"Expected {message_contains!r} in {str(exc)!r}"
            )
        return exc
    else:
        raise AssertionError(
            f"Expected class definition to fail with "
            f"{exception_type.__name__}"
        )


exc = assert_class_definition_fails(
    '''
class AnotherIllegal(Terminal):
    pass
''',
    TypeError,
    "Cannot subclass final class Terminal",
)

str(exc)


'Cannot subclass final class Terminal'

## 23. Debugging challenge — find the bug

### Problem

Why is this metaclass dangerous?

```python
class BrokenMeta(type):
    def __new__(cls, name, bases, namespace):
        namespace["tag"] = "broken"
        return type.__new__(type, name, bases, namespace)
```

Fix it.

### Solution

The call:

```python
type.__new__(type, ...)
```

asks `type` to create a normal `type` instance and may discard the custom metaclass identity.

The first argument to `type.__new__` should be the actual metaclass being instantiated, and cooperative `super()` is preferable.


In [27]:
class FixedMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        namespace = dict(namespace)
        namespace["tag"] = "fixed"
        return super().__new__(mcls, name, bases, namespace, **kwargs)


class FixedExample(metaclass=FixedMeta):
    pass


assert type(FixedExample) is FixedMeta
assert FixedExample.tag == "fixed"

type(FixedExample), FixedExample.tag


(__main__.FixedMeta, 'fixed')

## 24. Debugging challenge — returning the wrong object from metaclass `__new__`

### Problem

A metaclass `__new__` must normally return the newly created **class object**.

Demonstrate what happens if it returns an unrelated value.


In [28]:
class StrangeMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        return {"not": "a class"}


Strange = StrangeMeta("Strange", (), {})
Strange


{'not': 'a class'}

Directly calling `StrangeMeta(...)` returned a dictionary because the metaclass `__new__` chose to return one.

This illustrates how fundamental `__new__` is: it determines the object produced by construction.

In normal framework code, a metaclass `__new__` should return a proper class object unless highly unusual behavior is explicitly intended.


## 25. Challenge — count instances per class

### Problem

Create `InstanceCountingMeta`.

Every class using it should expose:

```python
ClassName.instances_created
```

The count should increment only when instance construction succeeds.

If `__init__` raises, the count must not increase.


In [29]:
class InstanceCountingMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        cls = super().__new__(mcls, name, bases, namespace, **kwargs)
        cls.instances_created = 0
        return cls

    def __call__(cls, *args, **kwargs):
        instance = super().__call__(*args, **kwargs)
        cls.instances_created += 1
        return instance


class Ticket(metaclass=InstanceCountingMeta):
    def __init__(self, valid=True):
        if not valid:
            raise ValueError("invalid ticket")


Ticket()
Ticket()
assert Ticket.instances_created == 2

try:
    Ticket(valid=False)
except ValueError:
    pass

assert Ticket.instances_created == 2
Ticket.instances_created


2

## 26. Challenge — normalize class constants

### Problem

Create `ConstantMeta`.

Any ALL_CAPS string value defined directly in the class body should be normalized by:

- trimming leading/trailing whitespace,
- converting it to uppercase.

Do not alter inherited constants and do not alter lowercase attributes.


In [30]:
class ConstantMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        namespace = dict(namespace)

        for key, value in list(namespace.items()):
            if (
                key.isupper()
                and isinstance(value, str)
                and not key.startswith("__")
            ):
                namespace[key] = value.strip().upper()

        return super().__new__(mcls, name, bases, namespace, **kwargs)


class Protocol(metaclass=ConstantMeta):
    METHOD = "  get "
    content_type = "application/json"


assert Protocol.METHOD == "GET"
assert Protocol.content_type == "application/json"

Protocol.METHOD, Protocol.content_type


('GET', 'application/json')

## 27. Challenge — enforce naming conventions

### Problem

Build `NamingMeta`.

Rules:

- concrete class names must end in `Service`,
- public methods must be snake_case,
- framework base class `ServiceBase` is exempt.

Use a small regular expression for snake_case.


In [31]:
import re

snake_case_re = re.compile(r"^[a-z][a-z0-9_]*$")


class NamingMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        if name != "ServiceBase" and not name.endswith("Service"):
            raise TypeError(f"{name} must end with 'Service'")

        for attr_name, value in namespace.items():
            if attr_name.startswith("_"):
                continue

            if callable(value) and not snake_case_re.fullmatch(attr_name):
                raise TypeError(
                    f"Public method {attr_name!r} must use snake_case"
                )

        return super().__new__(mcls, name, bases, namespace, **kwargs)


class ServiceBase(metaclass=NamingMeta):
    pass


class PaymentService(ServiceBase):
    def process_payment(self):
        return "ok"


assert PaymentService().process_payment() == "ok"

try:
    exec(
        '''
class BadName(ServiceBase):
    pass
''',
        globals(),
        {},
    )
except TypeError as exc:
    naming_error = str(exc)

naming_error


"BadName must end with 'Service'"

## 28. Challenge — add slots automatically

### Problem

Create `AutoSlotsMeta`.

If a class defines:

```python
fields = ("x", "y")
```

and does not define `__slots__`, automatically set:

```python
__slots__ = ("x", "y")
```

Demonstrate that instances then lack a normal `__dict__`.

Be careful: slot names cannot conflict with class variables of the same name.


In [32]:
class AutoSlotsMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        namespace = dict(namespace)
        fields = namespace.get("fields")

        if fields is not None and "__slots__" not in namespace:
            if not isinstance(fields, tuple):
                raise TypeError("fields must be a tuple")
            namespace["__slots__"] = fields

        return super().__new__(mcls, name, bases, namespace, **kwargs)


class SlotPoint(metaclass=AutoSlotsMeta):
    fields = ("x", "y")

    def __init__(self, x, y):
        self.x = x
        self.y = y


sp = SlotPoint(1, 2)
assert sp.x == 1 and sp.y == 2
assert not hasattr(sp, "__dict__")

(sp.x, sp.y, hasattr(sp, "__dict__"))


(1, 2, False)

## 29. Challenge — class-level computed metadata

### Problem

Create `SchemaMeta` that computes immutable metadata for each class:

```python
__field_names__
__required_fields__
```

Rules:

- annotations define all fields,
- annotated names without a class-level default are required,
- metadata is stored as tuples.

Example:

```python
class Product(...):
    id: int
    name: str
    price: float = 0.0
```

should have:

```python
__field_names__ == ("id", "name", "price")
__required_fields__ == ("id", "name")
```


In [33]:
class SchemaMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        annotations = dict(namespace.get("__annotations__", {}))

        field_names = tuple(annotations)
        required_fields = tuple(
            field_name
            for field_name in field_names
            if field_name not in namespace
        )

        namespace = dict(namespace)
        namespace["__field_names__"] = field_names
        namespace["__required_fields__"] = required_fields

        return super().__new__(mcls, name, bases, namespace, **kwargs)


class Product(metaclass=SchemaMeta):
    id: int
    name: str
    price: float = 0.0


assert Product.__field_names__ == ("id", "name", "price")
assert Product.__required_fields__ == ("id", "name")

Product.__field_names__, Product.__required_fields__


(('id', 'name', 'price'), ('id', 'name'))

## 30. Final synthesis problem — framework-quality metaclass

### Problem

Create a `RecordMeta` metaclass with all of the following behavior:

1. Fields come from type annotations.
2. Base-class fields are inherited.
3. A subclass may add fields.
4. The metaclass generates `__init__` if one is absent.
5. Unknown keyword arguments are rejected.
6. Missing required arguments are rejected.
7. Class-level defaults make fields optional.
8. The generated `__repr__` displays every field.
9. A user-defined `__repr__` must not be overwritten.
10. The metaclass must use cooperative `super()`.

Then create `EmployeeRecord` inheriting from `BaseRecord`.


In [34]:
class RecordMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        namespace = dict(namespace)

        inherited_annotations = {}
        inherited_defaults = {}

        for base in bases:
            inherited_annotations.update(
                getattr(base, "__record_annotations__", {})
            )
            inherited_defaults.update(
                getattr(base, "__record_defaults__", {})
            )

        local_annotations = dict(namespace.get("__annotations__", {}))

        all_annotations = {
            **inherited_annotations,
            **local_annotations,
        }

        all_defaults = dict(inherited_defaults)

        for field_name in local_annotations:
            if field_name in namespace:
                all_defaults[field_name] = namespace[field_name]

        namespace["__record_annotations__"] = all_annotations
        namespace["__record_defaults__"] = all_defaults

        fields = tuple(all_annotations)
        namespace["__record_fields__"] = fields

        if fields and "__init__" not in namespace:
            def __init__(self, **values):
                unknown = set(values) - set(fields)
                if unknown:
                    raise TypeError(
                        f"Unknown fields: {sorted(unknown)}"
                    )

                missing = [
                    field
                    for field in fields
                    if field not in values and field not in all_defaults
                ]
                if missing:
                    raise TypeError(
                        f"Missing required fields: {missing}"
                    )

                for field in fields:
                    if field in values:
                        value = values[field]
                    else:
                        value = all_defaults[field]
                    setattr(self, field, value)

            namespace["__init__"] = __init__

        if fields and "__repr__" not in namespace:
            def __repr__(self):
                body = ", ".join(
                    f"{field}={getattr(self, field)!r}"
                    for field in fields
                )
                return f"{type(self).__name__}({body})"

            namespace["__repr__"] = __repr__

        return super().__new__(mcls, name, bases, namespace, **kwargs)


class BaseRecord(metaclass=RecordMeta):
    id: int


class EmployeeRecord(BaseRecord):
    name: str
    department: str = "General"
    active: bool = True


employee = EmployeeRecord(id=101, name="Lin")

assert employee.id == 101
assert employee.name == "Lin"
assert employee.department == "General"
assert employee.active is True

repr(employee)


"EmployeeRecord(id=101, name='Lin', department='General', active=True)"

In [35]:
# Error checks for the synthesis problem

try:
    EmployeeRecord(name="Missing ID")
except TypeError as exc:
    missing_record_error = str(exc)

try:
    EmployeeRecord(id=1, name="X", surprise=True)
except TypeError as exc:
    unknown_record_error = str(exc)

missing_record_error, unknown_record_error


("Missing required fields: ['id']", "Unknown fields: ['surprise']")

# Best-practice checklist for custom metaclasses

Use this checklist in production code.

### Construction

- Prefer `super().__new__(mcls, name, bases, namespace, **kwargs)`.
- Accept and forward `**kwargs` unless you intentionally consume them.
- Return the class object created by the superclass.
- Avoid calling `type.__new__(type, ...)` when you mean to preserve a custom metaclass.

### Namespace handling

- Do not overwrite user-defined methods unless that is an explicit framework contract.
- Copy the namespace before large transformations if that improves clarity.
- Use `__prepare__` only when you need to control or observe class-body namespace behavior.

### Inheritance

- Think carefully about inherited configuration.
- Do not mutate mutable configuration objects owned by base classes.
- Use cooperative `super()` so composed metaclasses can work through MRO.

### Validation

- Fail early with specific `TypeError` / `ValueError` messages.
- Validate classes at class-creation time when invalid classes should never exist.
- Test both valid and invalid class definitions.

### API design

- Prefer class decorators for opt-in transformations.
- Prefer `__init_subclass__` for subclass registration/validation tied to a base class.
- Use a metaclass when class creation semantics genuinely need central control.

### Maintainability

- Keep metaclass behavior documented.
- Minimize surprising implicit mutation.
- Preserve function metadata when wrapping callables.
- Write tests for inheritance, multiple inheritance, overrides, and class keywords.


# Review questions

Try answering these without running code.

1. Why is `type(Circle)` equal to the metaclass rather than `Circle`'s base class?
2. What is the conceptual difference between metaclass `__new__` and metaclass `__call__`?
3. When is `__prepare__` necessary?
4. Why can metaclass conflicts occur in multiple inheritance?
5. Why should framework metaclasses generally use cooperative `super()`?
6. When would `__init_subclass__` be simpler than a metaclass?
7. Why should injected methods avoid overwriting names already present in the namespace?
8. Why can loop-generated functions suffer from late binding?
9. Why is it useful to validate classes during class creation?
10. How can inheriting from `ABCMeta` solve compatibility problems with abstract base classes?


# Compact answer key

1. A class object is an **instance of its metaclass**.
2. Metaclass `__new__` creates the class object; metaclass `__call__` controls calls to that class object, usually instance creation.
3. Use `__prepare__` when the class-body namespace itself needs special behavior, such as duplicate detection.
4. Python must find a metaclass compatible with all base-class metaclasses.
5. `super()` allows every compatible metaclass in the MRO to participate.
6. Use `__init_subclass__` when behavior only needs to run for subclasses of a particular base class.
7. Silent overwriting makes APIs surprising and can destroy intentional user behavior.
8. Closures capture variables, not frozen loop values; default arguments can freeze the current value.
9. It prevents invalid class definitions from entering the program and moves errors closer to their source.
10. A metaclass derived from `ABCMeta` remains compatible with the ABC machinery while adding custom behavior.
